# 第2章 ハンズオン：GGUF / llama.cpp で CPU 推論

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shintaro-Abe/llm-quantization-dojo/blob/main/notebooks/02_gguf_llama_cpp.ipynb)

| 項目 | 内容 |
| :-- | :-- |
| 所要時間 | 約 15〜25 分（無料 Colab **CPU**・初回はモデルDLを含む） |
| 必要ランタイム | 無料 Colab の **CPU** ランタイム（GPU不要） |
| 既定モデル | `HuggingFaceTB/SmolLM2-1.7B`（Apache-2.0 / 非ゲート / safetensors） |
| やること | llama.cpp を入手 → GGUF(16bit) へ変換 → `Q4_K_M` に量子化 → CPU で前後比較 |
| llama.cpp | **ビルド済バイナリ**（タグ **`b9743`**）を使用（コンパイル不要・軽量） |
| 最終確認日 | 2026-06-21 |

!!! note
    **このNotebookの検証状況**：作者により **CPU 実機（devコンテナ）で Run all 完走**を確認済みです（静的検証＋ロジック検証＋実完走）。
    所要時間・推論速度は環境のCPUコア数に依存します。記載の目安は控えめな想定値です。

## このNotebookでやること

第1章は **GPU でモデルを鍛える**話でした。第2章は、その先の **CPU で動かして配る**話です。

1. **llama.cpp を入手** … 変換・量子化・推論のツールを用意する（ビルド済バイナリを使うので待ち時間が短い）
2. **GGUF(16bit) へ変換** … Hugging Face のモデルを `llama.cpp` の形式にする
3. **`Q4_K_M` に量子化** … 4bit に圧縮してファイルを小さくする
4. **前後比較** … 16bit と 4bit で「サイズ・出力・速度」を並べて見る

!!! warning "ランタイムは CPU にしてください（GPU不要）"
    上部メニュー →「ランタイム」→「ランタイムのタイプを変更」→ **CPU** を選択。
    この章は CPU だけで完走します。

## セットアップ：llama.cpp を入手する

`llama.cpp` を**ビルド済みバイナリ**（公式 GitHub Releases）で入手します。コンパイル不要なので速く・軽く、CPUランタイムでも安定します。

内訳は次の2つです（どちらもタグ `b9743` に固定）：

- **変換スクリプト** `convert_hf_to_gguf.py` … リポジトリを浅くクローンして取得（ビルドはしない）
- **量子化・推論ツール** `llama-quantize` / `llama-completion` … 配布物（`*-bin-ubuntu-x64.tar.gz`）を展開して取得

In [ ]:
# 1) 変換スクリプトとその依存を入手（リポジトリは浅くクローンするだけ＝ビルドしない）
#    依存は %pip でカーネルと同じ環境に入れる（!pip だと別の python に入り後で import できないことがある）
!git clone --depth 1 --branch b9743 https://github.com/ggml-org/llama.cpp
%pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
%pip install -q -U huggingface_hub

In [ ]:
# 2) ビルド済みバイナリ（CPU / ubuntu-x64）をダウンロードして展開
TAG = 'b9743'
URL = f'https://github.com/ggml-org/llama.cpp/releases/download/{TAG}/llama-{TAG}-bin-ubuntu-x64.tar.gz'
!wget -q {URL} -O llama-prebuilt.tar.gz
!mkdir -p prebuilt && tar xzf llama-prebuilt.tar.gz -C prebuilt
print('downloaded & extracted prebuilt binaries for', TAG)

In [ ]:
# 実行ファイル・スクリプトの場所は配布物の構成で変わりうるため、find で実パスを解決しておく
import pathlib

def find_one(name, *roots):
    for root in roots:
        hits = [p for p in pathlib.Path(root).rglob(name) if p.is_file()]
        if hits:
            return str(hits[0])
    raise FileNotFoundError(f'{name} が見つかりません。前のセルが成功しているか確認してください。')

CONVERT = find_one('convert_hf_to_gguf.py', 'llama.cpp')
QUANTIZE = find_one('llama-quantize', 'prebuilt')
COMPLETION = find_one('llama-completion', 'prebuilt')
print('convert   :', CONVERT)
print('quantize  :', QUANTIZE)
print('completion:', COMPLETION)

## ステップ1：モデルを取得する

既定モデルは第1章と同じ `HuggingFaceTB/SmolLM2-1.7B`（Apache-2.0・非ゲート）。ログイン不要です。

In [ ]:
from huggingface_hub import snapshot_download

MODEL_ID = 'HuggingFaceTB/SmolLM2-1.7B'
model_dir = snapshot_download(MODEL_ID, local_dir='SmolLM2-1.7B')
print('downloaded to', model_dir)

## ステップ2：GGUF(16bit) へ変換する

まず量子化前の **16bit GGUF** を作ります。これが「量子化前」の比較基準になります。

変換スクリプトの場所はバージョンで変わるため、先ほど `find` で解決した `CONVERT` を使います。

In [ ]:
import os, sys, subprocess

# 16bit(f16) GGUF へ変換（フラグ名は b9743 で確認済み: --outtype f16）
# カーネルと同じ python(sys.executable)で実行し、失敗時はそのまま例外にする(check=True)
subprocess.run([sys.executable, CONVERT, 'SmolLM2-1.7B',
                '--outtype', 'f16', '--outfile', 'smollm2-f16.gguf'], check=True)

f16_bytes = os.path.getsize('smollm2-f16.gguf')
print(f'f16 GGUF size = {f16_bytes/1024**3:.2f} GiB ({f16_bytes:,} bytes)')

## ステップ3：`Q4_K_M` に量子化する

16bit GGUF を **4bit（`Q4_K_M`）** に量子化します。`Q4_K_M` は「軽さと質のバランス型」の定番です（記号の読み方は[座学](https://shintaro-abe.github.io/llm-quantization-dojo/chapters/02-gguf-llama-cpp/)を参照）。

In [ ]:
# Q4_K_M へ量子化（INPUT OUTPUT TYPE の順）。QUANTIZE はビルド済バイナリ
subprocess.run([QUANTIZE, 'smollm2-f16.gguf', 'smollm2-Q4_K_M.gguf', 'Q4_K_M'], check=True)

q4_bytes = os.path.getsize('smollm2-Q4_K_M.gguf')
print(f'Q4_K_M GGUF size = {q4_bytes/1024**3:.2f} GiB ({q4_bytes:,} bytes)')
print(f'縮小率 = {100*(1-q4_bytes/f16_bytes):.1f}%  （{f16_bytes/q4_bytes:.2f} 倍小さい）')

## ステップ4：CPU で推論して前後比較する

同じプロンプト・同じ設定（seed・温度0）で、**16bit** と **`Q4_K_M`** をそれぞれ推論します。
`llama.cpp` は生成速度（tokens/sec）を表示するので、それも拾って比べます。

> 補足：base モデルを温度0で長く生成すると同じ語の繰り返しになりがちなので、生成トークン数 `-n` に上限を付けます（デモではこれで十分）。

In [ ]:
import os, re, subprocess

PROMPT = 'The capital of France is'
N_PREDICT = 48
N_THREADS = os.cpu_count() or 2

def run_completion(model_path):
    """llama-completion で決定的に生成し、(本文, 生成tok/s) を返す。
    速度は stderr の common_perf timing から拾う。失敗してもサイズ・本文は出せるようにする。"""
    proc = subprocess.run(
        [COMPLETION, '-m', model_path, '-p', PROMPT,
         '-n', str(N_PREDICT), '-s', '42', '--temp', '0',
         '-t', str(N_THREADS), '--no-display-prompt'],
        stdin=subprocess.DEVNULL, capture_output=True, text=True,
    )
    body = proc.stdout.strip()
    tok_s = None
    for line in proc.stderr.splitlines():
        # 'eval time' の行（'prompt eval time' は除く）から tokens per second を拾う
        if 'eval time' in line and 'prompt eval' not in line:
            m = re.search(r'([0-9.]+)\s*tokens per second', line)
            if m:
                tok_s = float(m.group(1))
    return body, tok_s

f16_out, f16_tps = run_completion('smollm2-f16.gguf')
print('=== 16bit (f16) ===')
print(f16_out)
print(f'\n生成速度: {f16_tps} tok/s' if f16_tps else '\n生成速度: 取得できず（本文は上記）')

In [ ]:
q4_out, q4_tps = run_completion('smollm2-Q4_K_M.gguf')
print('=== 4bit (Q4_K_M) ===')
print(q4_out)
print(f'\n生成速度: {q4_tps} tok/s' if q4_tps else '\n生成速度: 取得できず（本文は上記）')

## まとめ：量子化の前後を1枚で見る

サイズ・速度・出力を並べて、量子化のトレードオフを確認します。

In [ ]:
def fmt_tps(x):
    return f'{x:.1f} tok/s' if x else 'N/A'

speedup = (q4_tps / f16_tps) if (f16_tps and q4_tps) else None

print('| 指標            | 16bit (f16)        | 4bit (Q4_K_M)      |')
print('| :--             | :--                | :--                |')
print(f'| ファイルサイズ   | {f16_bytes/1024**3:.2f} GiB          | {q4_bytes/1024**3:.2f} GiB          |')
print(f'| 生成速度(CPU)   | {fmt_tps(f16_tps):<18} | {fmt_tps(q4_tps):<18} |')
print()
print(f'→ サイズは約 {f16_bytes/q4_bytes:.1f} 倍小さく' + (f'、生成は約 {speedup:.1f} 倍速く' if speedup else '') + 'なりました。')
print('  出力（上のセル）を見比べて、4bit でも文章が破綻していないことを確認しましょう。')
print(f'  ※ この値は本ランタイム（{N_THREADS} スレッド）での実測です。コア数が少ない環境では遅くなります。')

## 振り返りと次へ

おつかれさまでした。この章で体験したこと：

- Hugging Face のモデルを **自分の手で GGUF 化・量子化**した
- **GPU なし（CPU）** で LLM を動かした
- 量子化で **サイズが約1/3**になり、それでも文章が成立することを確認した

### 理解度を確認する

- GGUF が「何のための形式」かを自分の言葉で説明できる
- 量子化の前後で「サイズ・出力・速度」がどう変わったかを説明できる
- 第1章（bitsandbytes）と第2章（GGUF）の使い分けを1文で言える

学んだことは GitHub Issue（章タスク）に記録しましょう（進め方は手順書で案内）。

!!! tip "発展：第1章で微調整したモデルを GGUF 化してみよう"
    第1章で作った『〜だミャ。』モデル（LoRAアダプタ）を PEFT の `merge_and_unload` で base にマージし、
    本Notebookと同じ手順で GGUF 化すれば、**あなたが鍛えたモデルを CPU で配布・推論**できます。総仕上げにどうぞ。

### 公式情報で裏取りする

- [第2章 公式リンク集（references）](https://shintaro-abe.github.io/llm-quantization-dojo/chapters/02-gguf-llama-cpp/references/)